# Classification Workflow
## Sample Pipeline for SDTM.DM domain

Importing key modules:

In [24]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects

Importing functions:

In [25]:
import functions

Setting the study of interest (Clinical Study B):

In [26]:
teststudy="PH_0218_A"

Initializing the dictionary where to store the unique instances in the training set for the source EDC datasets containing demographics data:

In [27]:
counts = {}

## Sample Workflow for SDTM.DM domain
### Training the model
Setting the list for the SDTM modules of interest:

In [28]:
list_of_modules = ['dm','ie']

Dictionaries to store the trained classifier, vectorizer and the metrics on the validation sets for each module are initialized.\
With compute_traindf function, all training data is read and concatenated into a single dataframe. Then weights for single variables are computed based on the computeWeights function and the 10x scaling factor. Last, build_NLPmodelWeights function is used to build the classifier, passing the computed weights.

In [29]:
classifiers = {}
checks = {}
vectorizers = {}

for mod in list_of_modules:
    unique_train_df = functions.compute_traindf(mod,'PheedIt',teststudy)
    counts[mod]=unique_train_df.shape[0]
    weights = functions.computeWeights(unique_train_df,scale='on')
    check,vectorizer,classifier=functions.build_NLPmodelWeights(unique_train_df,weights)
    classifiers[mod]=classifier
    checks[mod]=check
    vectorizers[mod]=vectorizer

    unique_train_df_supp = functions.compute_traindf(str('supp'+mod),'PheedIt',teststudy)
    counts['supp'+mod]=unique_train_df_supp.shape[0]
    weights = functions.computeWeights(unique_train_df_supp,scale='on')
    check,vectorizer,classifier=functions.build_NLPmodelWeights(unique_train_df_supp,weights)
    classifiers['supp'+mod]=classifier
    checks['supp'+mod]=check
    vectorizers['supp'+mod]=vectorizer


Checking the unique counts for the DM module:

In [30]:
counts['dm']

44

### Make the prediction
After building the model, the new and original variables for the Clinical Study B are predicted to map to corresponding SDTM variables, employing the same encoding vectorizer.\
Finally, metrics are computed based on the performance return by the classifier compared to the expert review mapping. 

In [31]:
metrics = {}
outputs = {}

for mod in list_of_modules:

    out = functions.predict_annot(mod,teststudy,'0218',"Input","Output",vectorizers[mod],classifiers[mod])
    outputs[mod]=out
    metrics[mod] = functions.check_teststudy(out)

    suppout = functions.predict_annot('supp'+mod,teststudy,'0218',"Input","Output",vectorizers['supp'+mod],classifiers['supp'+mod])
    outputs['supp'+mod]=suppout
    metrics['supp'+mod] = functions.check_teststudy(suppout)



m:\Philomed\Statistics\CDISC\Python\Article\functions.py:132: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out2[0]=out2[0].str.upper()
m:\Philomed\Statistics\CDISC\Python\Article\functions.py:133: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out2['pred'] = out['pred'].str.strip(' ')
m:\Philomed\Statistics\CDISC\Python\Article\functions.py:132: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats

In [32]:
outputs['dm']

,0,pred,real
0,study,STUDYID,STUDYID
1,module,DOMAIN,DOMAIN
2,visit,DROP,DROP
3,patno,SUBJID,SUBJID
4,page_no,DROP,DROP
5,rep_no,DROP,DROP
6,sex_n,DROP,DROP
7,dsstmo_n,RFICDTC2,RFICDTC2
8,dmmo_n,DMDTC2,DMDTC2
9,brthmo_n,BRTHDTC2,BRTHDTC2


In [33]:
metrics['dm']

,tp,fn,fp,tn,recall,specificity,accuracy,f1
0,16,0,1,9,1.0,0.9,0.961538,0.947368
